# Customer Churn Analysis - Feature Engineering

This notebook focuses on feature engineering and data preparation for machine learning models.

## Contents
1. Data Loading and Cleaning
2. Feature Creation
3. Feature Encoding
4. Feature Scaling
5. Feature Selection
6. Save Processed Data

In [ ]:
# Import libraries
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

from src.config import (
    RAW_DATA_PATH, PROCESSED_DATA_PATH,
    CATEGORICAL_FEATURES, NUMERICAL_FEATURES, TARGET,
    RANDOM_STATE, TEST_SIZE
)
from src.data_preprocessing import (
    load_data, clean_data, create_features,
    encode_features, prepare_data, preprocess_pipeline
)

# Settings
pd.set_option('display.max_columns', None)
sns.set_style('whitegrid')
%matplotlib inline

import warnings
warnings.filterwarnings('ignore')

## 1. Data Loading and Cleaning

In [ ]:
# Load raw data
df_raw = load_data()
print(f"Raw data shape: {df_raw.shape}")
df_raw.head()

In [ ]:
# Clean data
df_clean = clean_data(df_raw)
print(f"\nCleaned data shape: {df_clean.shape}")
print(f"Missing values: {df_clean.isnull().sum().sum()}")
df_clean.head()

In [ ]:
# Check data types after cleaning
print("\nData types after cleaning:")
print(df_clean.dtypes)

## 2. Feature Creation

We'll create several new features based on EDA insights:

In [ ]:
# Create new features
df_feat = create_features(df_clean)
print(f"\nData shape after feature creation: {df_feat.shape}")
print(f"New features added: {df_feat.shape[1] - df_clean.shape[1]}")

In [ ]:
# Display new features
new_features = [col for col in df_feat.columns if col not in df_clean.columns]
print(f"\nNew features created: {new_features}")
df_feat[new_features + [TARGET]].head(10)

In [ ]:
# Analyze new features
print("\nTenure Group Distribution:")
print(df_feat['tenure_group'].value_counts())

print("\nService Count Distribution:")
print(df_feat['service_count'].value_counts().sort_index())

print("\nAverage Monthly Charges Statistics:")
print(df_feat['avg_monthly_charges'].describe())

In [ ]:
# Visualize new features vs churn
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Tenure group vs churn
pd.crosstab(df_feat['tenure_group'], df_feat[TARGET], normalize='index').plot(
    kind='bar', ax=axes[0, 0], color=['steelblue', 'coral']
)
axes[0, 0].set_title('Churn Rate by Tenure Group', fontweight='bold')
axes[0, 0].set_xlabel('Tenure Group')
axes[0, 0].set_ylabel('Proportion')
axes[0, 0].legend(title='Churn')
axes[0, 0].tick_params(axis='x', rotation=45)

# Service count vs churn
pd.crosstab(df_feat['service_count'], df_feat[TARGET], normalize='index').plot(
    kind='bar', ax=axes[0, 1], color=['steelblue', 'coral']
)
axes[0, 1].set_title('Churn Rate by Service Count', fontweight='bold')
axes[0, 1].set_xlabel('Number of Services')
axes[0, 1].set_ylabel('Proportion')
axes[0, 1].legend(title='Churn')
axes[0, 1].tick_params(axis='x', rotation=0)

# Average monthly charges distribution by churn
df_feat[df_feat[TARGET] == 'No']['avg_monthly_charges'].hist(
    bins=30, alpha=0.5, color='blue', ax=axes[1, 0], label='No Churn'
)
df_feat[df_feat[TARGET] == 'Yes']['avg_monthly_charges'].hist(
    bins=30, alpha=0.5, color='red', ax=axes[1, 0], label='Churn'
)
axes[1, 0].set_title('Avg Monthly Charges Distribution', fontweight='bold')
axes[1, 0].set_xlabel('Avg Monthly Charges')
axes[1, 0].set_ylabel('Frequency')
axes[1, 0].legend()

# Long contract vs churn
pd.crosstab(df_feat['has_long_contract'], df_feat[TARGET], normalize='index').plot(
    kind='bar', ax=axes[1, 1], color=['steelblue', 'coral']
)
axes[1, 1].set_title('Churn Rate by Contract Type', fontweight='bold')
axes[1, 1].set_xlabel('Has Long Contract (1=Yes, 0=No)')
axes[1, 1].set_ylabel('Proportion')
axes[1, 1].legend(title='Churn')
axes[1, 1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

## 3. Feature Encoding

Convert categorical variables to numerical using one-hot encoding:

In [ ]:
# Encode features
df_encoded = encode_features(df_feat, drop_first=True)
print(f"\nData shape after encoding: {df_encoded.shape}")
print(f"Number of features: {df_encoded.shape[1]}")

In [ ]:
# Display encoded columns
print("\nEncoded columns:")
print(df_encoded.columns.tolist())

In [ ]:
# Check for any remaining missing values
print(f"\nMissing values after encoding: {df_encoded.isnull().sum().sum()}")

## 4. Feature Scaling

We'll prepare the data for scaling (will be done during model training):

In [ ]:
# Separate features and target
X, y = prepare_data(df_encoded)

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")
print(f"\nTarget distribution:")
print(y.value_counts(normalize=True))

In [ ]:
# Split data for analysis
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)

print(f"Training set size: {X_train.shape[0]} ({X_train.shape[0]/len(X)*100:.1f}%)")
print(f"Test set size: {X_test.shape[0]} ({X_test.shape[0]/len(X)*100:.1f}%)")
print(f"\nTraining set churn rate: {y_train.mean()*100:.2f}%")
print(f"Test set churn rate: {y_test.mean()*100:.2f}%")

In [ ]:
# Demonstrate scaling (will be done properly in model training)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("\nFeature statistics before scaling:")
print(pd.DataFrame(X_train).describe().loc[['mean', 'std']].round(2))

print("\nFeature statistics after scaling:")
print(pd.DataFrame(X_train_scaled, columns=X.columns).describe().loc[['mean', 'std']].round(2))

## 5. Feature Selection

Analyze feature importance and correlations:

In [ ]:
# Calculate correlation with target
feature_correlations = pd.DataFrame(X).corrwith(y).sort_values(ascending=False)

print("Top 15 features correlated with churn:")
print(feature_correlations.head(15))

print("\nBottom 15 features correlated with churn:")
print(feature_correlations.tail(15))

In [ ]:
# Visualize top correlations
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Top positive correlations
feature_correlations.head(15).plot(kind='barh', ax=axes[0], color='coral')
axes[0].set_title('Top 15 Positive Correlations with Churn', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Correlation Coefficient')
axes[0].invert_yaxis()

# Top negative correlations
feature_correlations.tail(15).plot(kind='barh', ax=axes[1], color='steelblue')
axes[1].set_title('Top 15 Negative Correlations with Churn', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Correlation Coefficient')
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

In [ ]:
# Check for multicollinearity
correlation_matrix = pd.DataFrame(X).corr()

# Find highly correlated features (> 0.8)
high_corr_pairs = []
for i in range(len(correlation_matrix.columns)):
    for j in range(i+1, len(correlation_matrix.columns)):
        if abs(correlation_matrix.iloc[i, j]) > 0.8:
            high_corr_pairs.append((
                correlation_matrix.columns[i],
                correlation_matrix.columns[j],
                correlation_matrix.iloc[i, j]
            ))

if high_corr_pairs:
    print("Highly correlated feature pairs (|r| > 0.8):")
    for pair in high_corr_pairs:
        print(f"  {pair[0]} <-> {pair[1]}: {pair[2]:.3f}")
else:
    print("No highly correlated features found (|r| > 0.8)")

## 6. Save Processed Data

In [ ]:
# Save processed data
df_encoded.to_csv(PROCESSED_DATA_PATH, index=False)
print(f"\nProcessed data saved to: {PROCESSED_DATA_PATH}")
print(f"Shape: {df_encoded.shape}")

In [ ]:
# Summary of feature engineering
print("\n" + "="*80)
print("FEATURE ENGINEERING SUMMARY")
print("="*80)
print(f"Original features: {df_raw.shape[1]}")
print(f"After cleaning: {df_clean.shape[1]}")
print(f"After feature creation: {df_feat.shape[1]}")
print(f"After encoding: {df_encoded.shape[1]}")
print(f"\nFinal feature count for modeling: {X.shape[1]}")
print(f"Total samples: {X.shape[0]}")
print(f"\nClass balance:")
print(f"  No Churn: {(y==0).sum()} ({(y==0).mean()*100:.1f}%)")
print(f"  Churn: {(y==1).sum()} ({(y==1).mean()*100:.1f}%)")

In [ ]:
print("\n" + "="*80)
print("FEATURE ENGINEERING COMPLETE")
print("="*80)
print("\nNext Steps:")
print("1. Model Training with Multiple Algorithms")
print("2. Hyperparameter Tuning")
print("3. Model Evaluation and Comparison")
print("4. SHAP Explainability Analysis")
print("\nProceed to notebook 03_modeling.ipynb")